# ⚡ LegacyNode — Kaggle GPU Cloud Runner

**Purpose:** Boot an Ollama server, load a Qwen Coder model, and expose it over a
Cloudflare reverse tunnel so LegacyNode can call it from your local machine.

## Setup Steps
1. Enable GPU accelerator: **Settings → Accelerator → GPU T4 x2**
2. Turn on Internet: **Settings → Internet → On**
3. Run All cells (Shift+Enter through each, or Session → Run All)
4. Copy the public Cloudflare URL printed in **Cell 4** into your local `.env` as `TUNNEL_URL`

## Models
Change `MODEL_NAME` below to switch models. Requires enough VRAM:
- `qwen2.5-coder:7b` — ~5 GB VRAM ✅ (T4 safe)
- `qwen2.5-coder:32b` — ~20 GB VRAM ⚠️ (needs T4 x2 or P100)
- `qwen2.5-coder:72b` — ~45 GB VRAM ❌ (needs A100)


In [ ]:
# ─── Configuration ──────────────────────────────────────────
MODEL_NAME   = "qwen2.5-coder:32b"   # Change to fit your GPU VRAM
OLLAMA_PORT  = 11434
TUNNEL_TOOL  = "cloudflare"           # 'cloudflare' | 'zrok' | 'ngrok'
ZROK_TOKEN   = ""                     # Only needed if TUNNEL_TOOL='zrok'
NGROK_TOKEN  = ""                     # Only needed if TUNNEL_TOOL='ngrok'
# ────────────────────────────────────────────────────────────
print(f"Config: model={MODEL_NAME}, port={OLLAMA_PORT}, tunnel={TUNNEL_TOOL}")

In [ ]:
# Cell 1 — Install Ollama
import subprocess, os, time

print("📦 Installing Ollama...")
result = subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True, capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else "")
if result.returncode != 0:
    print("STDERR:", result.stderr[-300:])
    raise RuntimeError("Ollama install failed")
print("✅ Ollama installed")

# Verify ollama binary
v = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
print("Version:", v.stdout.strip())

In [ ]:
# Cell 2 — Start Ollama Server
import subprocess, time, requests

print("🚀 Starting Ollama server...")
env = os.environ.copy()
env["OLLAMA_HOST"] = f"0.0.0.0:{OLLAMA_PORT}"
env["OLLAMA_ORIGINS"] = "*"  # Allow cross-origin requests from tunnel

server_proc = subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print(f"  PID: {server_proc.pid}")

# Wait for server to be ready
for i in range(30):
    time.sleep(1)
    try:
        r = requests.get(f"http://localhost:{OLLAMA_PORT}/api/tags", timeout=2)
        if r.status_code == 200:
            print(f"✅ Ollama ready after {i+1}s")
            break
    except Exception:
        pass
else:
    raise RuntimeError("Ollama server did not start in 30s")

In [ ]:
# Cell 3 — Pull Model
import subprocess

print(f"📥 Pulling model: {MODEL_NAME}")
print("   (This may take 10–30 min on first run — model weights ~20GB)")

result = subprocess.run(
    ["ollama", "pull", MODEL_NAME],
    capture_output=False   # Show download progress
)

if result.returncode != 0:
    raise RuntimeError(f"Failed to pull {MODEL_NAME}")
print(f"✅ Model {MODEL_NAME} ready")

# Quick sanity inference test
import requests, json
test = requests.post(
    f"http://localhost:{OLLAMA_PORT}/api/generate",
    json={"model": MODEL_NAME, "prompt": "Say 'OK' in one word.", "stream": False},
    timeout=60,
)
print("Inference test:", json.loads(test.text).get("response", "???").strip())

In [ ]:
# Cell 4 — Start Reverse Tunnel & Print Public URL
import subprocess, threading, time, re

tunnel_url = None

if TUNNEL_TOOL == "cloudflare":
    print("🌐 Starting Cloudflare trycloudflare tunnel...")
    # Install cloudflared
    subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && "
        "chmod +x /usr/local/bin/cloudflared",
        shell=True, check=True
    )
    # Launch tunnel and capture URL from stderr
    url_event = threading.Event()
    output_lines = []

    def _run_tunnel():
        global tunnel_url
        proc = subprocess.Popen(
            ["cloudflared", "tunnel", "--url", f"http://localhost:{OLLAMA_PORT}"],
            stderr=subprocess.PIPE, stdout=subprocess.PIPE, text=True
        )
        for line in proc.stderr:
            output_lines.append(line)
            match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
            if match:
                tunnel_url = match.group(0)
                url_event.set()

    t = threading.Thread(target=_run_tunnel, daemon=True)
    t.start()

    print("  Waiting for tunnel URL (up to 30s)...")
    found = url_event.wait(timeout=30)
    if not found:
        print("Last output:", "\n".join(output_lines[-5:]))
        raise RuntimeError("Cloudflare tunnel URL not found in 30s")

elif TUNNEL_TOOL == "zrok":
    print("🌐 Starting zrok tunnel...")
    subprocess.run(
        "curl -sSLf https://get.zrok.io | bash -s -- --no-install-key",
        shell=True, check=True
    )
    subprocess.run(["zrok", "enable", ZROK_TOKEN], check=True)
    proc = subprocess.Popen(
        ["zrok", "share", "public", f"localhost:{OLLAMA_PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    time.sleep(5)
    # Read URL from stdout
    for _ in range(10):
        line = proc.stdout.readline()
        m = re.search(r'https://[^\s]+\.zrok\.io', line)
        if m:
            tunnel_url = m.group(0)
            break

elif TUNNEL_TOOL == "ngrok":
    print("🌐 Starting ngrok tunnel...")
    subprocess.run(
        "pip install -q pyngrok && "
        f"python -c \"from pyngrok import ngrok; ngrok.set_auth_token('{NGROK_TOKEN}')\"",
        shell=True, check=True
    )
    from pyngrok import ngrok
    http_tunnel = ngrok.connect(OLLAMA_PORT, "http")
    tunnel_url = http_tunnel.public_url

# ─── Print the URL ──────────────────────────────────────────
print("\n" + "="*60)
print(f"  ✅ TUNNEL ACTIVE")
print(f"  PUBLIC URL : {tunnel_url}")
print(f"  MODEL      : {MODEL_NAME}")
print(f"  OLLAMA API : {tunnel_url}/v1")
print("="*60)
print("\n👉 Copy this into your local .env file:")
print(f"TUNNEL_URL={tunnel_url}")
print(f"LLM_MODEL={MODEL_NAME}")

In [ ]:
# Cell 5 — Keep-Alive Heartbeat
# Kaggle terminates idle sessions after ~20 minutes.
# This cell sends periodic requests to keep the kernel active.
import time, requests

print("💓 Keep-alive heartbeat started (Ctrl+C to stop)")
print(f"   Polling Ollama every 60s to prevent idle timeout")

while True:
    try:
        r = requests.get(f"http://localhost:{OLLAMA_PORT}/api/tags", timeout=5)
        status = "✅" if r.status_code == 200 else f"⚠️ {r.status_code}"
        ts = time.strftime("%H:%M:%S")
        print(f"  [{ts}] Ollama health: {status} | Tunnel: {tunnel_url}")
    except Exception as e:
        print(f"  Heartbeat error: {e}")
    time.sleep(60)